<a href="https://colab.research.google.com/github/amanpoonia/Research_project_Prof.Nour/blob/main/code_till_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.spatial.distance import jensenshannon, pdist, squareform
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Change this value to control the number of clusters used by all methods.
# For decision trees, this sets the maximum number of leaf clusters.
N_CLUSTERS = 6

def main():
    # 1. DATA ACQUISITION
    print("Downloading data...")
    data = yf.download('EURUSD=X', start='2003-12-01', end='2025-12-31')
    if data.empty:
        raise RuntimeError("Failed to download data.")

    close = data['Close'].copy()
    if isinstance(close, pd.DataFrame):
        close = close.squeeze()
    # 2. DATA PREPARATION
    print("Preparing data...")
    log_ret = np.log(close / close.shift(1))
    log_ret = log_ret.dropna()
    q30 = log_ret.quantile(0.3)
    q70 = log_ret.quantile(0.7)
    states = pd.Series(index=log_ret.index, dtype=int)
    states[log_ret < q30] = -1
    states[(log_ret >= q30) & (log_ret <= q70)] = 0
    states[log_ret > q70] = 1
    # 3. STATE CONSTRUCTION
    LOOKBACK = 5
    df = pd.DataFrame({'State': states})
    df['s1'] = df['State'].shift(5)
    df['s2'] = df['State'].shift(4)
    df['s3'] = df['State'].shift(3)
    df['s4'] = df['State'].shift(2)
    df['s5'] = df['State'].shift(1)
    df['Y'] = df['State']
    df = df.dropna().astype(int)
    # 4. STATE ORDERING
    order = []
    for s5 in [-1,0,1]:
        for s4 in [-1,0,1]:
            for s3 in [-1,0,1]:
                for s2 in [-1,0,1]:
                    for s1 in [-1,0,1]:
                        order.append((s1,s2,s3,s4,s5))
    # 5. TRANSITION TABLE
    print("Computing transition table...")
    grouped = df.groupby(['s1', 's2', 's3', 's4', 's5'])['Y'].value_counts().unstack(fill_value=0)
    for col in [-1, 0, 1]:
        if col not in grouped.columns:
            grouped[col] = 0
    grouped = grouped[[-1, 0, 1]]
    grouped = grouped.reindex(order, fill_value=0)
    N_Obs = grouped.sum(axis=1)
    P_neg1 = grouped[-1] / N_Obs
    P_0 = grouped[0] / N_Obs
    P_pos1 = grouped[1] / N_Obs
    P_neg1 = P_neg1.fillna(0)
    P_0 = P_0.fillna(0)
    P_pos1 = P_pos1.fillna(0)
    Direction = P_pos1 - P_neg1
    Confidence = np.maximum(np.maximum(P_neg1, P_0), P_pos1)
    Class_Error = 1 - Confidence
    Gini = P_neg1*(1-P_neg1) + P_0*(1-P_0) + P_pos1*(1-P_pos1)
    def safe_log(p):
        return np.log(p) if p > 0 else 0
    Cross_Entropy = -(P_neg1.apply(safe_log)*P_neg1 + P_0.apply(safe_log)*P_0 + P_pos1.apply(safe_log)*P_pos1)
    transition_table = pd.DataFrame({
        'N_Observations': N_Obs,
        'P(-1)': P_neg1,
        'P(0)': P_0,
        'P(+1)': P_pos1,
        'Direction': Direction,
        'Classification_Error': Class_Error,
        'Gini': Gini,
        'Cross_Entropy': Cross_Entropy,
        'Confidence': Confidence
    })
    transition_table.to_csv('state_transition_table.csv')
    # 6. METHODS 1 & 2: DECISION TREES
    X = df[['s1', 's2', 's3', 's4', 's5']]
    Y = df['Y']
    if N_CLUSTERS < 2:
        raise ValueError(f"N_CLUSTERS must be at least 2, got {N_CLUSTERS}.")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def build_fixed_cluster_tree(criterion):
        tree = DecisionTreeClassifier(
            criterion=criterion,
            max_leaf_nodes=N_CLUSTERS,
            random_state=42
        )
        score = cross_val_score(tree, X, Y, cv=cv, scoring='accuracy').mean()
        tree.fit(X, Y)
        return tree, tree.get_depth(), score

    print("Building fixed-cluster Gini Tree...")
    gini_tree, gini_depth, gini_acc = build_fixed_cluster_tree('gini')
    print(f"Gini depth: {gini_depth}")
    print(f"Gini accuracy: {gini_acc}")

    print("Building fixed-cluster Entropy Tree...")
    entropy_tree, entropy_depth, entropy_acc = build_fixed_cluster_tree('entropy')
    print(f"Entropy depth: {entropy_depth}")
    print(f"Entropy accuracy: {entropy_acc}")
    # 7. TREE CLUSTERS
    def process_tree_clusters(tree, prefix):
        leaf_ids = tree.apply(X)
        df_cluster = df.copy()
        df_cluster['Cluster'] = leaf_ids

        clust_grouped = df_cluster.groupby('Cluster')['Y'].value_counts().unstack(fill_value=0)
        for col in [-1, 0, 1]:
            if col not in clust_grouped.columns:
                clust_grouped[col] = 0
        clust_grouped = clust_grouped[[-1, 0, 1]]

        N_Obs = clust_grouped.sum(axis=1)
        P_neg1 = clust_grouped[-1] / N_Obs
        P_0 = clust_grouped[0] / N_Obs
        P_pos1 = clust_grouped[1] / N_Obs

        Direction = P_pos1 - P_neg1
        Confidence = np.maximum(np.maximum(P_neg1, P_0), P_pos1)
        Class_Error = 1 - Confidence
        Gini = P_neg1*(1-P_neg1) + P_0*(1-P_0) + P_pos1*(1-P_pos1)
        Cross_Entropy = -(P_neg1.apply(safe_log)*P_neg1 + P_0.apply(safe_log)*P_0 + P_pos1.apply(safe_log)*P_pos1)

        state_cols = ['s1', 's2', 's3', 's4', 's5']
        unique_states = df_cluster.groupby('Cluster')[state_cols].apply(lambda x: len(x.drop_duplicates()))

        summary = pd.DataFrame({
            'N_States': unique_states,
            'N_Observations': N_Obs,
            'P(-1)': P_neg1,
            'P(0)': P_0,
            'P(+1)': P_pos1,
            'Direction': Direction,
            'Classification_Error': Class_Error,
            'Gini': Gini,
            'Cross_Entropy': Cross_Entropy,
            'Confidence': Confidence
        })

        summary = summary.sort_values('Direction')
        summary.to_csv(f'{prefix}_tree_cluster_summary.csv')

        members = df_cluster[state_cols + ['Cluster']].drop_duplicates()
        order_map = {st: i for i, st in enumerate(order)}
        members['order_idx'] = members.apply(lambda row: order_map[tuple(row[state_cols])], axis=1)
        members = members.sort_values('order_idx').drop(columns=['order_idx'])
        members.to_csv(f'{prefix}_tree_cluster_members.csv', index=False)

        return summary
    gini_summary = process_tree_clusters(gini_tree, 'gini')
    print("\nGini clusters:")
    print(len(gini_summary))
    print(gini_summary[['N_States','N_Observations','Direction']])

    entropy_summary = process_tree_clusters(entropy_tree, 'entropy')
    print("\nEntropy clusters:")
    print(len(entropy_summary))
    print(entropy_summary[['N_States','N_Observations','Direction']])
    # 8. STATE CLUSTERING
    valid_states = transition_table[transition_table['N_Observations'] > 0].copy()
    state_probs = valid_states[['P(-1)', 'P(0)', 'P(+1)']].values
    if not 2 <= N_CLUSTERS <= len(state_probs):
        raise ValueError(
            f"N_CLUSTERS must be between 2 and {len(state_probs)}, "
            f"got {N_CLUSTERS}."
        )

    print("Jensen-Shannon Clustering...")
    js_dist = pdist(state_probs, metric=jensenshannon)
    js_dist = np.nan_to_num(js_dist, nan=0.0)
    js_square = squareform(js_dist)
    js_clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='precomputed', linkage='average')
    js_labels = js_clusterer.fit_predict(js_square)
    valid_states['JS_Cluster'] = js_labels

    print("SSE Clustering...")
    sse_clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='euclidean', linkage='average')
    sse_labels = sse_clusterer.fit_predict(state_probs)
    valid_states['SSE_Cluster'] = sse_labels
    def process_state_clusters(cluster_col, prefix):
        df_cluster = df.copy()
        df_cluster = df_cluster.join(valid_states[[cluster_col]], on=['s1','s2','s3','s4','s5'], how='inner')
        df_cluster = df_cluster.rename(columns={cluster_col: 'Cluster'})

        clust_grouped = df_cluster.groupby('Cluster')['Y'].value_counts().unstack(fill_value=0)
        for col in [-1, 0, 1]:
            if col not in clust_grouped.columns:
                clust_grouped[col] = 0
        clust_grouped = clust_grouped[[-1, 0, 1]]

        N_Obs = clust_grouped.sum(axis=1)
        P_neg1 = clust_grouped[-1] / N_Obs
        P_0 = clust_grouped[0] / N_Obs
        P_pos1 = clust_grouped[1] / N_Obs

        Direction = P_pos1 - P_neg1
        Confidence = np.maximum(np.maximum(P_neg1, P_0), P_pos1)
        Class_Error = 1 - Confidence
        Gini = P_neg1*(1-P_neg1) + P_0*(1-P_0) + P_pos1*(1-P_pos1)
        Cross_Entropy = -(P_neg1.apply(safe_log)*P_neg1 + P_0.apply(safe_log)*P_0 + P_pos1.apply(safe_log)*P_pos1)

        state_cols = ['s1', 's2', 's3', 's4', 's5']
        unique_states = df_cluster.groupby('Cluster')[state_cols].apply(lambda x: len(x.drop_duplicates()))

        summary = pd.DataFrame({
            'N_States': unique_states,
            'N_Observations': N_Obs,
            'P(-1)': P_neg1,
            'P(0)': P_0,
            'P(+1)': P_pos1,
            'Direction': Direction,
            'Classification_Error': Class_Error,
            'Gini': Gini,
            'Cross_Entropy': Cross_Entropy,
            'Confidence': Confidence
        })
        summary = summary.sort_values('Direction')
        summary.to_csv(f'{prefix}_cluster_summary.csv')

        members = df_cluster[state_cols + ['Cluster']].drop_duplicates()
        order_map = {st: i for i, st in enumerate(order)}
        members['order_idx'] = members.apply(lambda row: order_map[tuple(row[state_cols])], axis=1)
        members = members.sort_values('order_idx').drop(columns=['order_idx'])
        members.to_csv(f'{prefix}_cluster_members.csv', index=False)

        return summary
    js_summary = process_state_clusters('JS_Cluster', 'js')
    sse_summary = process_state_clusters('SSE_Cluster', 'sse')
    # 9. COMPARISON REPORT
    comparison = pd.DataFrame({
        'Method': ['Gini Tree', 'Cross-Entropy Tree', 'Jensen-Shannon', 'SSE'],
        'Tree_Depth': [gini_depth, entropy_depth, np.nan, np.nan],
        'CV_Accuracy': [gini_acc, entropy_acc, np.nan, np.nan],
        'Number_of_Clusters': [len(gini_summary), len(entropy_summary), len(js_summary), len(sse_summary)],
        'Average_Cluster_Size': [gini_summary['N_States'].mean(), entropy_summary['N_States'].mean(), js_summary['N_States'].mean(), sse_summary['N_States'].mean()],
        'Average_CrossEntropy': [gini_summary['Cross_Entropy'].mean(), entropy_summary['Cross_Entropy'].mean(), js_summary['Cross_Entropy'].mean(), sse_summary['Cross_Entropy'].mean()],
        'Average_Confidence': [gini_summary['Confidence'].mean(), entropy_summary['Confidence'].mean(), js_summary['Confidence'].mean(), sse_summary['Confidence'].mean()],
        'Average_Direction': [gini_summary['Direction'].mean(), entropy_summary['Direction'].mean(), js_summary['Direction'].mean(), sse_summary['Direction'].mean()],
        'Silhouette_Score': [np.nan, np.nan, np.nan, np.nan]
    })
    comparison.to_csv('comparison_report.csv', index=False)
    # 10. VISUALIZATIONS
    print("Generating visualizations...")
    plt.figure(figsize=(20,10))
    plot_tree(gini_tree, feature_names=['s1','s2','s3','s4','s5'], class_names=['-1','0','1'], filled=True)
    plt.title("Gini Decision Tree")
    plt.savefig('gini_tree_diagram.png')
    plt.close()
    plt.figure(figsize=(20,10))
    plot_tree(entropy_tree, feature_names=['s1','s2','s3','s4','s5'], class_names=['-1','0','1'], filled=True)
    plt.title("Cross-Entropy Decision Tree")
    plt.savefig('entropy_tree_diagram.png')
    plt.close()
    from scipy.cluster.hierarchy import dendrogram, linkage
    plt.figure(figsize=(15, 8))
    Z_js = linkage(js_dist, method='average')
    dendrogram(Z_js)
    plt.title("Jensen-Shannon Dendrogram")
    plt.savefig('js_dendrogram.png')
    plt.close()
    plt.figure(figsize=(15, 8))
    Z_sse = linkage(state_probs, method='average', metric='euclidean')
    dendrogram(Z_sse)
    plt.title("SSE Dendrogram")
    plt.savefig('sse_dendrogram.png')
    plt.close()
    def plot_heatmap(summary, title, filename):
        plt.figure(figsize=(8, 6))
        sns.heatmap(summary[['P(-1)','P(0)','P(+1)']], annot=True, cmap='RdYlGn', center=0.33)
        plt.title(title)
        plt.savefig(filename)
        plt.close()
    plot_heatmap(gini_summary, "Gini Tree Cluster Transition Probabilities", 'gini_heatmap.png')
    plot_heatmap(entropy_summary, "Entropy Tree Cluster Transition Probabilities", 'entropy_heatmap.png')
    plot_heatmap(js_summary, "JS Cluster Transition Probabilities", 'js_heatmap.png')
    plot_heatmap(sse_summary, "SSE Cluster Transition Probabilities", 'sse_heatmap.png')
    def plot_size(summary, title, filename):
        plt.figure(figsize=(8, 6))
        sns.barplot(x=summary.index, y=summary['N_States'], palette='viridis')
        plt.title(title)
        plt.xlabel('Cluster (Ordered by Direction)')
        plt.ylabel('Number of States')
        plt.savefig(filename)
        plt.close()
    plot_size(gini_summary, "Gini Tree Cluster Sizes", 'gini_size.png')
    plot_size(entropy_summary, "Entropy Tree Cluster Sizes", 'entropy_size.png')
    plot_size(js_summary, "JS Cluster Sizes", 'js_size.png')
    plot_size(sse_summary, "SSE Cluster Sizes", 'sse_size.png')
    def plot_direction(summary, title, filename):
        plt.figure(figsize=(8, 6))
        sns.barplot(x=summary.index, y=summary['Direction'], palette='coolwarm')
        plt.title(title)
        plt.xlabel('Cluster')
        plt.ylabel('Direction (P(+1) - P(-1))')
        plt.savefig(filename)
        plt.close()
    plot_direction(gini_summary, "Gini Tree Cluster Direction", 'gini_direction.png')
    plot_direction(entropy_summary, "Entropy Tree Cluster Direction", 'entropy_direction.png')
    plot_direction(js_summary, "JS Cluster Direction", 'js_direction.png')
    plot_direction(sse_summary, "SSE Cluster Direction", 'sse_direction.png')
    print("All done!")
if __name__ == '__main__':
    main()


[*********************100%***********************]  1 of 1 completed


Preparing data...
Computing transition table...
Building fixed-cluster Gini Tree...
Gini depth: 4
Gini accuracy: 0.39594634623018904
Building fixed-cluster Entropy Tree...
Entropy depth: 4
Entropy accuracy: 0.3950716096130943

Gini clusters:
6
         N_States  N_Observations  Direction
Cluster                                     
8               9             137  -0.218978
7              18             381  -0.047244
9              54            1607  -0.011201
1              81            1719   0.009889
10             27             680   0.017647
5              54            1199   0.028357

Entropy clusters:
6
         N_States  N_Observations  Direction
Cluster                                     
8               9             137  -0.218978
7              18             381  -0.047244
3              81            2287  -0.002624
9              54            1154   0.000867
10             27             565   0.028319
5              54            1199   0.028357
Jensen-Shannon 

CODE till s0 where s0 is day t-6

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.spatial.distance import jensenshannon, pdist, squareform
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Change this value to control the number of clusters used by all methods.
# For decision trees, this sets the maximum number of leaf clusters.
N_CLUSTERS = 6

def main():
    # 1. DATA ACQUISITION
    print("Downloading data...")
    data = yf.download('EURUSD=X', start='2003-12-01', end='2025-12-31')
    if data.empty:
        raise RuntimeError("Failed to download data.")

    close = data['Close'].copy()
    if isinstance(close, pd.DataFrame):
        close = close.squeeze()
    # 2. DATA PREPARATION
    print("Preparing data...")
    log_ret = np.log(close / close.shift(1))
    log_ret = log_ret.dropna()
    q30 = log_ret.quantile(0.3)
    q70 = log_ret.quantile(0.7)
    states = pd.Series(index=log_ret.index, dtype=int)
    states[log_ret < q30] = -1
    states[(log_ret >= q30) & (log_ret <= q70)] = 0
    states[log_ret > q70] = 1
    # 3. STATE CONSTRUCTION
    LOOKBACK = 5
    df = pd.DataFrame({'State': states})
    df['s0'] = df['State'].shift(6)                                         #t-6 (change)
    df['s1'] = df['State'].shift(5)
    df['s2'] = df['State'].shift(4)
    df['s3'] = df['State'].shift(3)
    df['s4'] = df['State'].shift(2)
    df['s5'] = df['State'].shift(1)
    df['Y'] = df['State']
    df = df.dropna().astype(int)
    # 4. STATE ORDERING
    order = []
    for s5 in [-1,0,1]:
        for s4 in [-1,0,1]:
            for s3 in [-1,0,1]:
                for s2 in [-1,0,1]:
                    for s1 in [-1,0,1]:
                        order.append((s1,s2,s3,s4,s5))
    # 5. TRANSITION TABLE
    print("Computing transition table...")
    grouped = df.groupby(['s1', 's2', 's3', 's4', 's5'])['Y'].value_counts().unstack(fill_value=0)
    for col in [-1, 0, 1]:
        if col not in grouped.columns:
            grouped[col] = 0
    grouped = grouped[[-1, 0, 1]]
    grouped = grouped.reindex(order, fill_value=0)
    N_Obs = grouped.sum(axis=1)
    P_neg1 = grouped[-1] / N_Obs
    P_0 = grouped[0] / N_Obs
    P_pos1 = grouped[1] / N_Obs
    P_neg1 = P_neg1.fillna(0)
    P_0 = P_0.fillna(0)
    P_pos1 = P_pos1.fillna(0)
    Direction = P_pos1 - P_neg1
    Confidence = np.maximum(np.maximum(P_neg1, P_0), P_pos1)
    Class_Error = 1 - Confidence
    Gini = P_neg1*(1-P_neg1) + P_0*(1-P_0) + P_pos1*(1-P_pos1)
    def safe_log(p):
        return np.log(p) if p > 0 else 0
    Cross_Entropy = -(P_neg1.apply(safe_log)*P_neg1 + P_0.apply(safe_log)*P_0 + P_pos1.apply(safe_log)*P_pos1)
    transition_table = pd.DataFrame({
        'N_Observations': N_Obs,
        'P(-1)': P_neg1,
        'P(0)': P_0,
        'P(+1)': P_pos1,
        'Direction': Direction,
        'Classification_Error': Class_Error,
        'Gini': Gini,
        'Cross_Entropy': Cross_Entropy,
        'Confidence': Confidence
    })
    transition_table.to_csv('state_transition_table.csv')
    # 6. METHODS 1 & 2: DECISION TREES
    X = df[['s1', 's2', 's3', 's4', 's5']]
    Y = df['Y']
    if N_CLUSTERS < 2:
        raise ValueError(f"N_CLUSTERS must be at least 2, got {N_CLUSTERS}.")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def build_fixed_cluster_tree(criterion):
        tree = DecisionTreeClassifier(
            criterion=criterion,
            min_samples_leaf=150,
            random_state=42
        )
        score = cross_val_score(tree, X, Y, cv=cv, scoring='accuracy').mean()
        tree.fit(X, Y)
        return tree, tree.get_depth(), score

    print("Building fixed-cluster Gini Tree...")
    gini_tree, gini_depth, gini_acc = build_fixed_cluster_tree('gini')
    print(f"Gini depth: {gini_depth}")
    print(f"Gini accuracy: {gini_acc}")

    print("Building fixed-cluster Entropy Tree...")
    entropy_tree, entropy_depth, entropy_acc = build_fixed_cluster_tree('entropy')
    print(f"Entropy depth: {entropy_depth}")
    print(f"Entropy accuracy: {entropy_acc}")
    # 7. TREE CLUSTERS
    def process_tree_clusters(tree, prefix):
        leaf_ids = tree.apply(X)
        df_cluster = df.copy()
        df_cluster['Cluster'] = leaf_ids

        clust_grouped = df_cluster.groupby('Cluster')['Y'].value_counts().unstack(fill_value=0)
        for col in [-1, 0, 1]:
            if col not in clust_grouped.columns:
                clust_grouped[col] = 0
        clust_grouped = clust_grouped[[-1, 0, 1]]

        N_Obs = clust_grouped.sum(axis=1)
        P_neg1 = clust_grouped[-1] / N_Obs
        P_0 = clust_grouped[0] / N_Obs
        P_pos1 = clust_grouped[1] / N_Obs

        Direction = P_pos1 - P_neg1
        Confidence = np.maximum(np.maximum(P_neg1, P_0), P_pos1)
        Class_Error = 1 - Confidence
        Gini = P_neg1*(1-P_neg1) + P_0*(1-P_0) + P_pos1*(1-P_pos1)
        Cross_Entropy = -(P_neg1.apply(safe_log)*P_neg1 + P_0.apply(safe_log)*P_0 + P_pos1.apply(safe_log)*P_pos1)

        state_cols = ['s1', 's2', 's3', 's4', 's5']
        unique_states = df_cluster.groupby('Cluster')[state_cols].apply(lambda x: len(x.drop_duplicates()))

        summary = pd.DataFrame({
            'N_States': unique_states,
            'N_Observations': N_Obs,
            'P(-1)': P_neg1,
            'P(0)': P_0,
            'P(+1)': P_pos1,
            'Direction': Direction,
            'Classification_Error': Class_Error,
            'Gini': Gini,
            'Cross_Entropy': Cross_Entropy,
            'Confidence': Confidence
        })

        summary = summary.sort_values('Direction')
        summary.to_csv(f'{prefix}_tree_cluster_summary.csv')

        members = df_cluster[state_cols + ['Cluster']].drop_duplicates()
        order_map = {st: i for i, st in enumerate(order)}
        members['order_idx'] = members.apply(lambda row: order_map[tuple(row[state_cols])], axis=1)
        members = members.sort_values('order_idx').drop(columns=['order_idx'])
        members.to_csv(f'{prefix}_tree_cluster_members.csv', index=False)

        return summary
    gini_summary = process_tree_clusters(gini_tree, 'gini')
    print("\nGini clusters:")
    print(len(gini_summary))
    print(gini_summary[['N_States','N_Observations','Direction']])

    entropy_summary = process_tree_clusters(entropy_tree, 'entropy')

    entropy_leaf_ids = entropy_tree.apply(X)                                  #change
    df_entropy = df.copy()
    df_entropy['Cluster'] = entropy_leaf_ids
    print("\nEntropy clusters:")
    print(len(entropy_summary))
    print(entropy_summary[['N_States','N_Observations','Direction']])




    # =====================================================
    # S0 ANALYSIS WITHIN ENTROPY CLUSTERS
    # =====================================================

    print("\nAnalyzing day-6 information (s0)...")

    s0_results = []

    def calc_entropy(series):

        counts = series.value_counts()

        probs = counts / counts.sum()

        probs = probs[probs > 0]

        return -(probs * np.log(probs)).sum()


    for cluster_id in sorted(df_entropy['Cluster'].unique()):

        cluster_df = df_entropy[
            df_entropy['Cluster'] == cluster_id
        ]

        parent_entropy = calc_entropy(
            cluster_df['Y']
        )

        weighted_child_entropy = 0

        for s0_value in [-1, 0, 1]:

            child_df = cluster_df[
                cluster_df['s0'] == s0_value
            ]

            if len(child_df) == 0:
                continue

            child_entropy = calc_entropy(
                child_df['Y']
            )

            weight = len(child_df) / len(cluster_df)

            weighted_child_entropy += (
                weight * child_entropy
            )

        information_gain = (
            parent_entropy
            - weighted_child_entropy
        )

        s0_results.append({
            'Cluster': cluster_id,
            'N_Observations': len(cluster_df),
            'Parent_Entropy': parent_entropy,
            'Child_Entropy': weighted_child_entropy,
            'Information_Gain': information_gain
        })

    s0_results = pd.DataFrame(s0_results)

    s0_results = s0_results.sort_values(
        'Information_Gain',
        ascending=False
    )

    print("\nS0 Information Gain:")
    print(s0_results)

    s0_results.to_csv(
        'entropy_cluster_s0_information_gain.csv',
        index=False
    )

    # =====================================================
    # S0 TRANSITION PROFILES
    # =====================================================

    rows = []

    for cluster_id in sorted(df_entropy['Cluster'].unique()):

        cluster_df = df_entropy[
            df_entropy['Cluster'] == cluster_id
        ]

        for s0_value in [-1,0,1]:

            temp = cluster_df[
                cluster_df['s0'] == s0_value
            ]

            if len(temp) == 0:
                continue

            counts = temp['Y'].value_counts()

            total = len(temp)

            p_neg = counts.get(-1,0)/total
            p_zero = counts.get(0,0)/total
            p_pos = counts.get(1,0)/total

            rows.append({
                'Cluster': cluster_id,
                's0': s0_value,
                'N_Observations': total,
                'P(-1)': p_neg,
                'P(0)': p_zero,
                'P(+1)': p_pos,
                'Direction': p_pos - p_neg
            })

    s0_profile = pd.DataFrame(rows)

    s0_profile.to_csv(
        'entropy_cluster_s0_transition_profiles.csv',
        index=False
    )

    print("\nSaved:")
    print("entropy_cluster_s0_transition_profiles.csv")







    # 8. STATE CLUSTERING
    valid_states = transition_table[transition_table['N_Observations'] > 0].copy()
    state_probs = valid_states[['P(-1)', 'P(0)', 'P(+1)']].values
    if not 2 <= N_CLUSTERS <= len(state_probs):
        raise ValueError(
            f"N_CLUSTERS must be between 2 and {len(state_probs)}, "
            f"got {N_CLUSTERS}."
        )

    print("Jensen-Shannon Clustering...")
    js_dist = pdist(state_probs, metric=jensenshannon)
    js_dist = np.nan_to_num(js_dist, nan=0.0)
    js_square = squareform(js_dist)
    js_clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='precomputed', linkage='average')
    js_labels = js_clusterer.fit_predict(js_square)
    valid_states['JS_Cluster'] = js_labels

    print("SSE Clustering...")
    sse_clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='euclidean', linkage='average')
    sse_labels = sse_clusterer.fit_predict(state_probs)
    valid_states['SSE_Cluster'] = sse_labels
    def process_state_clusters(cluster_col, prefix):
        df_cluster = df.copy()
        df_cluster = df_cluster.join(valid_states[[cluster_col]], on=['s1','s2','s3','s4','s5'], how='inner')
        df_cluster = df_cluster.rename(columns={cluster_col: 'Cluster'})

        clust_grouped = df_cluster.groupby('Cluster')['Y'].value_counts().unstack(fill_value=0)
        for col in [-1, 0, 1]:
            if col not in clust_grouped.columns:
                clust_grouped[col] = 0
        clust_grouped = clust_grouped[[-1, 0, 1]]

        N_Obs = clust_grouped.sum(axis=1)
        P_neg1 = clust_grouped[-1] / N_Obs
        P_0 = clust_grouped[0] / N_Obs
        P_pos1 = clust_grouped[1] / N_Obs

        Direction = P_pos1 - P_neg1
        Confidence = np.maximum(np.maximum(P_neg1, P_0), P_pos1)
        Class_Error = 1 - Confidence
        Gini = P_neg1*(1-P_neg1) + P_0*(1-P_0) + P_pos1*(1-P_pos1)
        Cross_Entropy = -(P_neg1.apply(safe_log)*P_neg1 + P_0.apply(safe_log)*P_0 + P_pos1.apply(safe_log)*P_pos1)

        state_cols = ['s1', 's2', 's3', 's4', 's5']
        unique_states = df_cluster.groupby('Cluster')[state_cols].apply(lambda x: len(x.drop_duplicates()))

        summary = pd.DataFrame({
            'N_States': unique_states,
            'N_Observations': N_Obs,
            'P(-1)': P_neg1,
            'P(0)': P_0,
            'P(+1)': P_pos1,
            'Direction': Direction,
            'Classification_Error': Class_Error,
            'Gini': Gini,
            'Cross_Entropy': Cross_Entropy,
            'Confidence': Confidence
        })
        summary = summary.sort_values('Direction')
        summary.to_csv(f'{prefix}_cluster_summary.csv')

        members = df_cluster[state_cols + ['Cluster']].drop_duplicates()
        order_map = {st: i for i, st in enumerate(order)}
        members['order_idx'] = members.apply(lambda row: order_map[tuple(row[state_cols])], axis=1)
        members = members.sort_values('order_idx').drop(columns=['order_idx'])
        members.to_csv(f'{prefix}_cluster_members.csv', index=False)

        return summary
    js_summary = process_state_clusters('JS_Cluster', 'js')
    sse_summary = process_state_clusters('SSE_Cluster', 'sse')
    # 9. COMPARISON REPORT
    comparison = pd.DataFrame({
        'Method': ['Gini Tree', 'Cross-Entropy Tree', 'Jensen-Shannon', 'SSE'],
        'Tree_Depth': [gini_depth, entropy_depth, np.nan, np.nan],
        'CV_Accuracy': [gini_acc, entropy_acc, np.nan, np.nan],
        'Number_of_Clusters': [len(gini_summary), len(entropy_summary), len(js_summary), len(sse_summary)],
        'Average_Cluster_Size': [gini_summary['N_States'].mean(), entropy_summary['N_States'].mean(), js_summary['N_States'].mean(), sse_summary['N_States'].mean()],
        'Average_CrossEntropy': [gini_summary['Cross_Entropy'].mean(), entropy_summary['Cross_Entropy'].mean(), js_summary['Cross_Entropy'].mean(), sse_summary['Cross_Entropy'].mean()],
        'Average_Confidence': [gini_summary['Confidence'].mean(), entropy_summary['Confidence'].mean(), js_summary['Confidence'].mean(), sse_summary['Confidence'].mean()],
        'Average_Direction': [gini_summary['Direction'].mean(), entropy_summary['Direction'].mean(), js_summary['Direction'].mean(), sse_summary['Direction'].mean()],
        'Silhouette_Score': [np.nan, np.nan, np.nan, np.nan]
    })
    comparison.to_csv('comparison_report.csv', index=False)
    # 10. VISUALIZATIONS
    print("Generating visualizations...")
    plt.figure(figsize=(20,10))
    plot_tree(gini_tree, feature_names=['s1','s2','s3','s4','s5'], class_names=['-1','0','1'], filled=True)
    plt.title("Gini Decision Tree")
    plt.savefig('gini_tree_diagram.png')
    plt.close()
    plt.figure(figsize=(20,10))
    plot_tree(entropy_tree, feature_names=['s1','s2','s3','s4','s5'], class_names=['-1','0','1'], filled=True)
    plt.title("Cross-Entropy Decision Tree")
    plt.savefig('entropy_tree_diagram.png')
    plt.close()
    from scipy.cluster.hierarchy import dendrogram, linkage
    plt.figure(figsize=(15, 8))
    Z_js = linkage(js_dist, method='average')
    dendrogram(Z_js)
    plt.title("Jensen-Shannon Dendrogram")
    plt.savefig('js_dendrogram.png')
    plt.close()
    plt.figure(figsize=(15, 8))
    Z_sse = linkage(state_probs, method='average', metric='euclidean')
    dendrogram(Z_sse)
    plt.title("SSE Dendrogram")
    plt.savefig('sse_dendrogram.png')
    plt.close()
    def plot_heatmap(summary, title, filename):
        plt.figure(figsize=(8, 6))
        sns.heatmap(summary[['P(-1)','P(0)','P(+1)']], annot=True, cmap='RdYlGn', center=0.33)
        plt.title(title)
        plt.savefig(filename)
        plt.close()
    plot_heatmap(gini_summary, "Gini Tree Cluster Transition Probabilities", 'gini_heatmap.png')
    plot_heatmap(entropy_summary, "Entropy Tree Cluster Transition Probabilities", 'entropy_heatmap.png')
    plot_heatmap(js_summary, "JS Cluster Transition Probabilities", 'js_heatmap.png')
    plot_heatmap(sse_summary, "SSE Cluster Transition Probabilities", 'sse_heatmap.png')
    def plot_size(summary, title, filename):
        plt.figure(figsize=(8, 6))
        sns.barplot(x=summary.index, y=summary['N_States'], palette='viridis')
        plt.title(title)
        plt.xlabel('Cluster (Ordered by Direction)')
        plt.ylabel('Number of States')
        plt.savefig(filename)
        plt.close()
    plot_size(gini_summary, "Gini Tree Cluster Sizes", 'gini_size.png')
    plot_size(entropy_summary, "Entropy Tree Cluster Sizes", 'entropy_size.png')
    plot_size(js_summary, "JS Cluster Sizes", 'js_size.png')
    plot_size(sse_summary, "SSE Cluster Sizes", 'sse_size.png')
    def plot_direction(summary, title, filename):
        plt.figure(figsize=(8, 6))
        sns.barplot(x=summary.index, y=summary['Direction'], palette='coolwarm')
        plt.title(title)
        plt.xlabel('Cluster')
        plt.ylabel('Direction (P(+1) - P(-1))')
        plt.savefig(filename)
        plt.close()
    plot_direction(gini_summary, "Gini Tree Cluster Direction", 'gini_direction.png')
    plot_direction(entropy_summary, "Entropy Tree Cluster Direction", 'entropy_direction.png')
    plot_direction(js_summary, "JS Cluster Direction", 'js_direction.png')
    plot_direction(sse_summary, "SSE Cluster Direction", 'sse_direction.png')
    print("All done!")
if __name__ == '__main__':
    main()


[*********************100%***********************]  1 of 1 completed


Preparing data...
Computing transition table...
Building fixed-cluster Gini Tree...
Gini depth: 7
Gini accuracy: 0.3937432436559074
Building fixed-cluster Entropy Tree...
Entropy depth: 7
Entropy accuracy: 0.3935685711668244

Gini clusters:
27
         N_States  N_Observations  Direction
Cluster                                     
17              9             181  -0.110497
7               9             151  -0.105960
43              9             190  -0.105263
44              9             187  -0.096257
30              6             184  -0.086957
52             18             332  -0.078313
25             12             324  -0.058642
46              9             193  -0.036269
31              6             288  -0.027778
5               9             153  -0.013072
35              9             198   0.005051
11              9             286   0.006993
24              6             153   0.013072
32              6             201   0.014925
36              9             278   